# Notebook to build code for Recommender Inference

## Recommender Model

In [1]:
import mlflow
import pandas as pd
import numpy as np
import joblib
import ast
from typing import Dict, List
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tensorflow_recommenders as tfrs

class RecommendationEngine:
    """Simple interface for getting fashion recommendations"""
    
    def __init__(self, model_path: str, articles_csv_path: str = '/data/processed/articles_df.csv',
                scaler_path: str = '/data/scalers/age_scaler_portugal.pkl', max_history: int = 50):
        """
        Initialize the recommendation engine.
        
        Args:
            model_path: Path to saved TensorFlow model
            articles_csv_path: Path to articles catalog CSV
            scaler_path: Path to age scaler pickle file
            max_history: Maximum purchase history length (must match training)
        """

        print(f"Loading model from {model_path}...")
        self.model = tf.saved_model.load(model_path)
        
        print(f"Loading articles catalog...")
        self.articles_df = pd.read_csv(articles_csv_path)
        self.articles_df['article_id'] = self.articles_df['article_id'].astype(str)
        
        print(f"Loading age scaler...")
        self.age_scaler = joblib.load(scaler_path)
        
        self.max_history = max_history
        print(f"Engine ready! {len(self.articles_df)} articles available.")
    
    def _pad_or_truncate(self, items: List[str]) -> List[str]:
        items = [str(x) for x in list(items)][:self.max_history]
        if len(items) < self.max_history:
            items = items + [""] * (self.max_history - len(items))
        return items

    
    def _ensure_list(self, x) -> List[str]:
        """Convert purchase history to list format"""
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    
    def recommend(self, customer_data: Dict, top_k: int = 5, exclude_purchased: bool = True) -> pd.DataFrame:
        """
        Get top-K recommendations for a customer.
        
        Args:
            customer_data: Dictionary with customer information:
                - customer_id: str (required)
                - age: int or float (required)
                - club_member_status: str (optional, e.g., "ACTIVE")
                - fashion_news_frequency: str (optional, e.g., "Regularly")
                - favorite_sales_channel: str (optional, e.g., "1")
                - purchased_articles: list of article IDs (optional)
            
            top_k: Number of recommendations to return
            exclude_purchased: Whether to exclude already purchased items
        
        Returns:
            DataFrame with top-K recommended articles and their details
        
        Example:
            >>> customer = {
            ...     'customer_id': 'C001',
            ...     'age': 28,
            ...     'club_member_status': 'ACTIVE',
            ...     'fashion_news_frequency': 'Regularly',
            ...     'favorite_sales_channel': '1',
            ...     'purchased_articles': ['0706016001', '0568601006']
            ... }
            >>> recs = engine.recommend(customer, top_k=5)
        """
        # Prepare customer features
        age_scaled = self.age_scaler.transform([[customer_data['age']]])[0][0]
        purchase_history = self._ensure_list(customer_data.get('purchased_articles', []))
        history_padded = self._pad_or_truncate(purchase_history)
        
        # Create input tensors
        input_tensors = {
            'customer_id': tf.constant([str(customer_data['customer_id'])], dtype=tf.string),
            'age_scaled': tf.constant([age_scaled], dtype=tf.float32),
            'club_member_status': tf.constant([str(customer_data.get('club_member_status', 'unknown'))], dtype=tf.string),
            'fashion_news_frequency': tf.constant([str(customer_data.get('fashion_news_frequency', 'unknown'))], dtype=tf.string),
            'favorite_sales_channel': tf.constant([str(customer_data.get('favorite_sales_channel', 'UNKNOWN'))], dtype=tf.string),
            'history_article_ids': tf.constant([history_padded], dtype=tf.string),
        }
        
        # Get customer embedding
        customer_embedding = self.model.customer_model(input_tensors, training=False)
        
        # Compute similarities with all articles
        all_scores = self._compute_article_scores(customer_embedding)
        
        # Get top candidates
        num_candidates = top_k * 2 if exclude_purchased else top_k
        top_indices = np.argsort(all_scores)[::-1][:num_candidates]
        recommended_article_ids = self.articles_df.iloc[top_indices]['article_id'].tolist()
        
        # Filter out purchased items if needed
        if exclude_purchased:
            purchased_set = set(purchase_history)
            recommended_article_ids = [
                aid for aid in recommended_article_ids 
                if aid not in purchased_set
            ][:top_k]
        
        # Get article details
        recommendations = self.articles_df[
            self.articles_df['article_id'].isin(recommended_article_ids)
        ].copy()
        
        # Preserve recommendation order
        recommendations['rank'] = recommendations['article_id'].map(
            {aid: i+1 for i, aid in enumerate(recommended_article_ids)}
        )
        recommendations = recommendations.sort_values('rank')
        
        return recommendations[[
            'rank', 'article_id', 'product_type_name', 
            'section_name', 'product_group_name', 'detail_desc'
        ]].reset_index(drop=True)
    
    def _compute_article_scores(self, customer_embedding: tf.Tensor) -> np.ndarray:
        """Compute similarity scores for all articles"""
        batch_size = 2048
        all_scores = []
        
        for i in range(0, len(self.articles_df), batch_size):
            batch_df = self.articles_df.iloc[i:i+batch_size]
            
            batch_inputs = {
                'article_id': tf.constant(batch_df['article_id'].values, dtype=tf.string),
                'product_type_name': tf.constant(batch_df['product_type_name'].fillna('unknown').astype(str).values, dtype=tf.string),
                'section_name': tf.constant(batch_df['section_name'].fillna('unknown').astype(str).values, dtype=tf.string),
                'product_group_name': tf.constant(batch_df['product_group_name'].fillna('unknown').astype(str).values, dtype=tf.string),
                'detail_desc': tf.constant(batch_df['detail_desc'].fillna('unknown').astype(str).values, dtype=tf.string),
            }
            
            article_embeddings = self.model.article_model(batch_inputs, training=False)
            batch_scores = tf.reduce_sum(customer_embedding * article_embeddings, axis=1)
            all_scores.append(batch_scores.numpy())
        
        return np.concatenate(all_scores)


2026-01-30 18:40:27.007497: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def get_recommendations(model_path: str, customer_data: Dict,
                        articles_csv_path: str = '/data/processed/articles_df.csv',
                        scaler_path: str = '/data/scalers/age_scaler_portugal.pkl',
                        top_k: int = 5, exclude_purchased: bool = True) -> pd.DataFrame:
    
    """
    Simple function to get recommendations without creating an engine instance.
    
    Args:
        model_path: Path to saved model
        customer_data: Customer information dict
        articles_csv_path: Path to articles CSV
        scaler_path: Path to age scaler
        top_k: Number of recommendations
        exclude_purchased: Filter out purchased items
    
    Returns:
        DataFrame with recommendations
    """

    engine = RecommendationEngine(model_path, articles_csv_path, scaler_path)
    return engine.recommend(customer_data, top_k, exclude_purchased)


## Mongo Data Retrieval

In [3]:
from pymongo import MongoClient
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)

def get_customer_info(customer_id: int, db: str = "stylistai", collection: str = "customers") -> dict:
    client = MongoClient(config['MONGO_CONNECTION_STRING'])
    db = client[db]
    customers_collection = db[collection]
    return customers_collection.find_one({ "customer_id": customer_id})

In [4]:
customer_info = get_customer_info(customer_id="00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657")
customer_info

{'_id': ObjectId('696925401d2d99038350974e'),
 'customer_id': '00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657',
 'Active': 0.0,
 'age': 49,
 'club_member_status': 'ACTIVE',
 'fashion_news_frequency': 'NONE',
 'favorite_sales_channel': 2.0,
 'postal_code': '974 04',
 'purchased_articles': [625548001,
  176209023,
  627759010,
  697138006,
  568601006,
  568601006,
  607642008,
  745232001,
  656719005,
  797065001,
  797065001,
  785186005,
  694736004,
  785710001,
  812683013,
  841260003,
  887593002,
  890498002,
  795440001,
  859416011,
  568601043]}

## Inference

In [5]:
MODEL_PATH = "/data/models/mlruns_outputs/fc49224e1ba24094ae6b606de6a86b17/saved_model"

print("\n" + "-"*60)
print("Customer Recommendation")
print("-"*60)

customer_info = get_customer_info(customer_id="00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657")

print(f"\nCustomer: {customer_info['customer_id']}")
print(f"Age: {customer_info['age']}, Status: {customer_info['club_member_status']}")
print(f"Past purchases: {len(customer_info['purchased_articles'])} items")

#recs = engine.recommend(customer, top_k=5)
recs = get_recommendations(model_path=MODEL_PATH, customer_data=customer_info, top_k=5)

print(f"\nTop 5 Recommendations:")
print("-" * 60)
for _, row in recs.iterrows():
    print(f"{row['rank']}. {row['article_id']} - {row['product_type_name']}")
    print(f"   {row['section_name']} / {row['product_group_name']}")
    print(f"   {row['detail_desc'][:70]}...")
    print()



------------------------------------------------------------
Customer Recommendation
------------------------------------------------------------

Customer: 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657
Age: 49, Status: ACTIVE
Past purchases: 21 items
Loading model from /data/models/mlruns_outputs/fc49224e1ba24094ae6b606de6a86b17/saved_model...


I0000 00:00:1769798433.910333    8943 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13713 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Loading articles catalog...
Loading age scaler...
Engine ready! 105542 articles available.


/home/dev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



Top 5 Recommendations:
------------------------------------------------------------
1. 694344001 - Underwear bottom
   Womens Lingerie / Underwear
   Shaping Brazilian briefs in microfibre and lace with a high waist, lin...

2. 610787002 - Pumps
   Womens Shoes / Shoes
   Court shoes in imitation leather with pointed toes and braided ties th...

3. 851370004 - Top
   Ladies H&M Sport / Garment Upper body
   Fully lined sports bra in fast-drying functional fabric with a gentle ...

4. 742100001 - Jacket
   Womens Trend / Garment Upper body
   Calf-length coat woven in a soft, patterned Tencel™ lyocell blend with...

5. 713183001 - Bikini top
   Womens Swimwear, beachwear / Swimwear
   Lined balconette bikini top with underwired, lightly padded cups, laci...



In [7]:
recs.to_dict()

{'rank': {0: 1, 1: 2, 2: 3, 3: 4, 4: 5},
 'article_id': {0: '694344001',
  1: '610787002',
  2: '851370004',
  3: '742100001',
  4: '713183001'},
 'product_type_name': {0: 'Underwear bottom',
  1: 'Pumps',
  2: 'Top',
  3: 'Jacket',
  4: 'Bikini top'},
 'section_name': {0: 'Womens Lingerie',
  1: 'Womens Shoes',
  2: 'Ladies H&M Sport',
  3: 'Womens Trend',
  4: 'Womens Swimwear, beachwear'},
 'product_group_name': {0: 'Underwear',
  1: 'Shoes',
  2: 'Garment Upper body',
  3: 'Garment Upper body',
  4: 'Swimwear'},
 'detail_desc': {0: 'Shaping Brazilian briefs in microfibre and lace with a high waist, lined gusset and half-string back. The briefs have a light shaping effect on the tummy and hips.',
  1: 'Court shoes in imitation leather with pointed toes and braided ties that can be wound around the ankles. Satin linings, imitation leather insoles and rubber soles. Covered heels 8 cm.',
  2: 'Fully lined sports bra in fast-drying functional fabric with a gentle V-neck and cut-out deta

## Once APi is built

In [2]:
import requests

url = "http://stylistai-api:8000/invoke"
payload = {
    "customer_id": "00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657",
    "top_k": 5,
    "exclude_purchased": True,
}

resp = requests.post(url, json=payload, timeout=30)
resp.raise_for_status()
print(resp.json()[0])


{'rank': 1, 'article_id': '694344001', 'product_type_name': 'Underwear bottom', 'section_name': 'Womens Lingerie', 'product_group_name': 'Underwear', 'detail_desc': 'Shaping Brazilian briefs in microfibre and lace with a high waist, lined gusset and half-string back. The briefs have a light shaping effect on the tummy and hips.'}
